# Part 2: Vector store + LLM action plans

Run **`RAG_part1_build_vector_store.ipynb`** first so `RAG_docs/vector_store/` exists (FAISS index + `rag_parents.json`).

**This notebook:** loads the **saved vector store only** (no re-reading `RAG_docs/knowledge/`). Retrieval uses chunk embeddings plus parent text from the parent store. **Shared helpers:** `utils/rag_utils.py`.

In [ ]:
import os
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

from utils.rag_utils import (
    balance_vector_hits_by_source_file,
    get_dominant_party_info,
    get_party_to_tier_mapping,
    load_attack_and_agentic,
    load_parent_store,
    load_predictions,
    load_vector_store,
    save_action_plan,
    save_comparison_file,
)

VECTOR_STORE_DIR = Path("RAG_docs/vector_store")
PREDICTIONS_DIR = Path("RAG_docs/predictions")
RESULTS_DIR = Path("RAG_docs/action_plans")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
vector_store = load_vector_store(VECTOR_STORE_DIR)
predictions_data = load_predictions(PREDICTIONS_DIR, verbose=True)
attack_actions_data, agentic_features_data = load_attack_and_agentic(verbose=True)

n_attack = (
    len(attack_actions_data.get("attacks", {}))
    if isinstance(attack_actions_data, dict)
    else 0
)
has_agentic = isinstance(agentic_features_data, dict) and any(
    t in agentic_features_data for t in ("RAN", "Edge", "Core")
)
print("-" * 60)
print(
    f"Predictions: {len(predictions_data)} | "
    f"Vector store: {'loaded' if vector_store is not None else 'missing'}"
)
print(f"Attack types in attack_options: {n_attack}")
print(f"Agentic features config: {'yes' if has_agentic else 'no'}")
print("-" * 60)

In [ ]:
# RAG retrieval + LLM query/prompts/pipeline (search lives here next to query construction)

_RAG_PARENTS = load_parent_store(VECTOR_STORE_DIR)


def search_rag_knowledge(
    vector_store: Any,
    query: str,
    top_k: int = 10,
    expand_parent_context: bool = True,
    max_parent_chars: int = 12000,
    balance_by_source_file: bool = True,
    fetch_oversample: int = 6,
) -> List[Dict[str, Any]]:
    """Similarity search over the saved index; optional fair quota per source_file metadata."""
    if vector_store is None:
        return []
    fetch_k = min(300, max(top_k, top_k * fetch_oversample))
    vector_results = vector_store.similarity_search_with_score(query, k=fetch_k)
    if balance_by_source_file:
        vector_results = balance_vector_hits_by_source_file(vector_results, top_k)
    else:
        vector_results = vector_results[:top_k]
    formatted_results: List[Dict[str, Any]] = []
    for doc, dist in vector_results:
        meta = doc.metadata or {}
        title = meta.get("retrieval_title") or meta.get("title", "Unknown")
        pid = meta.get("parent_id")
        src = meta.get("source_file", "")

        if expand_parent_context and pid and pid in _RAG_PARENTS:
            ptext = _RAG_PARENTS[pid].get("text") or ""
            if len(ptext) > max_parent_chars:
                ptext = ptext[:max_parent_chars] + "\n\n[parent truncated]"
            rt = _RAG_PARENTS[pid].get("retrieval_title") or title
            full_text = f"{rt}\n\n{ptext}"
        else:
            full_text = doc.page_content or meta.get("text", "")

        similarity_score = 1.0 / (1.0 + float(dist)) if dist > 0 else 1.0
        formatted_results.append(
            {
                "title": title,
                "text": full_text,
                "score": similarity_score,
                "source_file": src,
            }
        )
    return formatted_results


def build_llm_based_rag_query(sample: Dict[str, Any]) -> str:
    label = sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"
    confidence = sample.get("confidence", 0.0)
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample)
    shap_expl = sample.get("shap_explanation", {}) or {}
    party_contributions_pct = shap_expl.get("party_contributions_pct", {}) or {}
    feat_contribs = shap_expl.get("feature_contributions", {}) or {}
    party_to_tier = get_party_to_tier_mapping(sample)

    tier_features_summary: Dict[str, List[str]] = {t: [] for t in ["RAN", "Edge", "Core"]}
    for party_name, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        tier = party_to_tier.get(party_name, "")
        if not tier or tier not in tier_features_summary:
            continue
        party_feat_list = []
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            pct_contrib = float(v.get("pct_contribution", 0.0) or 0.0)
            abs_shap = float(v.get("abs_shap_value", 0.0) or 0.0)
            if pct_contrib > 0.0:
                party_feat_list.append((feat_name, pct_contrib, abs_shap))
        party_feat_list.sort(key=lambda x: x[1], reverse=True)
        tier_features_summary[tier].extend(
            [f"{name} ({pct:.2%})" for name, pct, _ in party_feat_list[:5]]
        )

    query_prompt = f"""Generate a concise RAG search query to find relevant mitigation actions for a cybersecurity attack prediction.

Prediction Details:
- Attack Type: {label}
- Confidence: {confidence:.1%}
- Dominant Network Tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)

Party Contributions by Network Tier:"""

    for party_name, contribution_pct in party_contributions_pct.items():
        if contribution_pct and float(contribution_pct) > 0:
            tier = party_to_tier.get(party_name, "Unknown")
            query_prompt += f"\n  - {party_name} ({tier}): {float(contribution_pct):.2%}"

    query_prompt += "\n\nTop Contributing Features by Network Tier:"
    for tier in ["RAN", "Edge", "Core"]:
        if tier_features_summary.get(tier):
            features_text = ", ".join(tier_features_summary[tier][:5])
            query_prompt += f"\n  - {tier}: {features_text}"

    query_prompt += f"""

Generate a search query (1-2 sentences) that will help retrieve relevant mitigation actions and strategies for this {label} attack, focusing on the {dominant_tier} network tier and the contributing features mentioned above.

Query:"""

    try:
        api_key = os.getenv("OPENAI_API_KEY")
        if api_key:
            client = OpenAI(api_key=api_key)
            response = client.chat.completions.create(
                model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
                messages=[
                    {
                        "role": "system",
                        "content": "You are a cybersecurity expert. Generate concise, effective search queries for retrieving mitigation actions.",
                    },
                    {"role": "user", "content": query_prompt},
                ],
                temperature=0.3,
                max_tokens=100,
            )
            return response.choices[0].message.content.strip()
        return (
            f"What mitigation actions are required for {label} attack in {dominant_tier} "
            f"network tier with {dominant_pct:.1f}% contribution?"
        )
    except Exception:
        return f"What mitigation actions are required for {label} attack in {dominant_tier} network tier?"


def create_prompt(
    sample_data: Dict[str, Any],
    rag_results: List[Dict[str, Any]],
    attack_actions_data: Optional[Dict[str, Any]] = None,
    agentic_features_data: Optional[Dict[str, Any]] = None,
) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    dominant_party, dominant_tier, dominant_pct = get_dominant_party_info(sample_data)

    attack_actions_context = ""
    if attack_actions_data and "attacks" in attack_actions_data:
        attack_type = predicted_label.upper()
        if attack_type in attack_actions_data["attacks"]:
            recommended_actions = attack_actions_data["attacks"][attack_type]
            attack_actions_context = (
                f"\n\nAttack-Specific Recommended Actions (from attack_options.json):\n"
                f"For {predicted_label} attack, recommended actions: {', '.join(recommended_actions)}\n"
            )
        else:
            attack_actions_context = (
                f"\n\nAttack-Specific Actions: No specific recommendations for {predicted_label}.\n"
            )

    agentic_context = ""
    if agentic_features_data:
        agentic_context = (
            "\n\nAgentic Features and Actions by Network Tier (from agentic_features.json):\n"
        )
        for tier in ["RAN", "Edge", "Core"]:
            if tier in agentic_features_data:
                tier_data = agentic_features_data[tier]
                features = tier_data.get("features", [])
                actions = tier_data.get("actions", [])
                agentic_context += f"\n{tier} Network Tier:\n"
                agentic_context += f"  - Available features: {len(features)} total features\n"
                agentic_context += f"  - Available actions: {', '.join(actions)}\n"

    top_5_results = rag_results[:5]
    rag_context = ""
    if top_5_results:
        rag_context = "\n\nKnowledge Base Context (from RAG search):\n"
        for idx, result in enumerate(top_5_results, 1):
            rag_context += f"\n[{idx}] {result['title']}\n"
            rag_context += f"{result['text'][:1000]}\n"
    else:
        rag_context = "\n\nKnowledge Base: No relevant documents found from RAG search."

    network_tier_info = ""
    if dominant_tier:
        network_tier_info = f"\n- Dominant network tier: {dominant_tier} (contribution: {dominant_pct:.1f}%)"
        if dominant_party:
            network_tier_info += f"\n- Dominant party: {dominant_party}"

    return f"""You are a cybersecurity decision-making agent specialized in attack response orchestration.
        Your role is NOT to invent mitigations, but to SELECT and ASSIGN actions from a predefined
        action set using explainability signals and agentic features.

        =====================
        INPUT CONTEXT
        =====================

        Prediction summary:
        - sample_id: {sample_id}
        - predicted_label: {predicted_label}
        - confidence: {confidence:.1%}

        Explainability & agentic evidence:
        - Party-level contributions and dominance:
        {network_tier_info}

        - Feature-level evidence (SHAP / statistics / signals):
        {str(sample_data)}

        Allowed actions (STRICT CONSTRAINT):
        {attack_actions_context}
        • Only actions listed above are allowed.
        • Do NOT invent, rename, generalize, or merge actions.

        Agentic decision signals:
        {agentic_context}

        Retrieved knowledge (RAG – optional support, may be empty):
        {rag_context}

        =====================
        TASK
        =====================

        Using ONLY the information above, generate an agent-ready action plan that:

        1) Interprets how {predicted_label} manifests across network tiers (RAN, Edge, Core).
        2) Identifies which evidence party MUST trigger mitigation first.
        3) Selects actions ONLY from the provided Allowed actions list.
        4) Assigns each action to the MOST appropriate executing party and network tier.
        5) Provides explicit, evidence-grounded reasoning for EACH action.
        6) Adapts aggressiveness and execution priority based on confidence ({confidence:.1%}).
        7) If no allowed action is suitable, return empty action lists.

        =====================
        OUTPUT FORMAT (STRICT)
        =====================

        Return a VALID JSON object ONLY:

        {{
          "threat_level": "Critical|High|Medium|Low",

          "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
          ],

          "primary_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken ",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],

          "supporting_actions": [
            {{
              "action": "EXACT action name from Allowed actions to be taken",
              "network_tier": "RAN|Edge|Core",
              "party_evidence_type": "type of evidence this party observed",
              "reasoning": "why this action supports or complements the primary action"
            }}
          ],
          "overall_reasoning": "Concise summary explaining party prioritization, tier ordering, and action selection logic",
          "execution_priority": "Immediate|High|Standard|Low",
          "knowledge_sources_used": [
            "allowed_actions_context",
            "attack_actions_context"
          ]
        }}

        =====================
        HARD RULES (DO NOT VIOLATE)
        =====================

        - Do NOT output text outside the JSON.
        - Do NOT generate actions not listed in Allowed actions.
        - The "all_actions" list MUST be the union of primary_actions and supporting_actions.
        - Do NOT alter action or party names.
        - Every action MUST include explicit reasoning tied to evidence or agentic rules.
        - Prefer dominant party and tier for primary actions unless contradicted by evidence.
        - If RAG context is empty, rely ONLY on explainability and agentic context.
        """


def create_prompt_without_RAG(sample_data: Dict[str, Any]) -> str:
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)

    return f"""You are a cybersecurity expert responsible for selecting mitigation actions
      based on attack predictions and domain knowledge.

      Your objective is to decide WHAT actions should be taken and WHY,
      strictly using a predefined set of allowed actions.

      =====================
      INPUT
      =====================

      Prediction summary:
      - sample_id: {sample_id}
      - predicted_label: {predicted_label}
      - confidence: {confidence:.1%}

      =====================
      TASK
      =====================

      Using ONLY the information above:

      1) Assess the severity of the predicted attack ({predicted_label})
        and assign an appropriate threat level.
      2) Select all relevant mitigation actions within 2 or 3 words
      3) Ensure selected actions are appropriate for the predicted attack type.
      4) Adapt action selection and urgency based on confidence ({confidence:.1%}).
      5) If no action is applicable, return an empty action list.

      =====================
      OUTPUT (STRICT JSON ONLY)
      =====================

      {{
        "threat_level": "Critical|High|Medium|Low",
        "all_actions": [
            "action_name_1",
            "action_name_2",
            "action_name_3"
        ],
        "primary_actions": [
            {{
              "action": "any action name  to be taken",
              "network_tier": "RAN|Edge|Core",
              "reasoning": "clear explanation linking evidence + agentic signals to this action"
            }}
          ],
        "overall_reasoning": "Brief explanation describing why these actions were selected and how confidence influenced the decision",
        "execution_priority": "Immediate|High|Standard|Low",
        "knowledge_sources_used": [
          "publicly_available_online_sources"
        ]
      }}

      =====================
      HARD RULES
      =====================

      - Do NOT output text outside the JSON.
      - Do NOT invent, rename, or generalize actions.
      - Use short, standard mitigation action names (2–3 words).
      - If no action applies, return empty lists for `all_actions` and `primary_actions`.
      - Keep reasoning concise and decision-focused.
      """


def call_llm_api(prompt: str) -> Optional[Dict[str, Any]]:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None

    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {
                "role": "system",
                "content": "You are a cybersecurity expert. Return only valid JSON.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
    )

    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find("{")
        end = response_text.rfind("}") + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")

    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": [],
    }


def run_agent_pipeline_for_sample(
    sample: Dict[str, Any],
    vector_store: Any,
    attack_actions_data: Optional[Dict[str, Any]],
    agentic_features_data: Optional[Dict[str, Any]],
    results_action_dir: Path,
    *,
    top_k_retrieve: int = 10,
    verbose: bool = True,
) -> None:
    if verbose:
        print("=" * 80)
        print(f"Processing Sample ID: {sample.get('sample_id')}")
        print(f"Predicted Label: {sample.get('predicted_label', 'UNKNOWN')}")
        print(f"Confidence: {sample.get('confidence', 0):.1%}")

    query = build_llm_based_rag_query(sample)
    if verbose:
        print("\n--- RAG query sent to vector search ---")
        print(query)
        print()

    rag_results = search_rag_knowledge(vector_store, query, top_k=top_k_retrieve)
    top5 = rag_results[:5]
    if verbose:
        print(f"--- Retrieved {len(rag_results)} docs; top 5 used in prompt ---")
        for idx, result in enumerate(top5, 1):
            snippet_len = 350
            text_preview = (result.get("text", "") or "")[:snippet_len]
            if len(result.get("text", "") or "") > snippet_len:
                text_preview += "..."
            print(
                f"\n  [{idx}] {result.get('title', 'Unknown')} "
                f"(similarity: {result.get('score', 0):.2%})"
            )
            print(f"      {text_preview}")
        if not top5:
            print("  (no documents retrieved)")
        print()

    prompt = create_prompt(
        sample, rag_results, attack_actions_data, agentic_features_data
    )
    prompt_without_rag = create_prompt_without_RAG(sample)

    if verbose:
        print("--- PROMPT SENT TO LLM (with RAG) ---")
        print(prompt)
        print("--- END PROMPT ---")
        print("\nCalling LLM API with RAG...")
    llm_response_with_rag = call_llm_api(prompt)

    if verbose:
        print("\nCalling LLM API without RAG...")
    llm_response_without_rag = call_llm_api(prompt_without_rag)

    output_file_with_rag: Optional[Path] = None
    if llm_response_with_rag:
        r = llm_response_with_rag
        if verbose:
            print("\n--- LLM response (with RAG) ---")
            print(f"  Threat level: {r.get('threat_level')}")
            print(f"  Execution priority: {r.get('execution_priority')}")
            print(f"  Primary actions: {r.get('primary_actions', [])}")
            reasoning = (r.get("overall_reasoning") or "")[:500]
            if len(r.get("overall_reasoning") or "") > 500:
                reasoning += "..."
            print(f"  Overall reasoning: {reasoning}")
        try:
            output_file_with_rag = save_action_plan(
                sample, query, rag_results, llm_response_with_rag, results_action_dir
            )
            if verbose:
                print(f"Saved with-RAG result to {output_file_with_rag.name}")
        except Exception as e:
            if verbose:
                print(f"Error saving with-RAG result: {e}")
            output_file_with_rag = None
    elif verbose:
        print("Failed to get LLM response with RAG")

    without_rag_path: Optional[Path] = None
    if llm_response_without_rag:
        r = llm_response_without_rag
        if verbose:
            print("\n--- LLM response (without RAG) ---")
            print(f"  Threat level: {r.get('threat_level')}")
            print(f"  Execution priority: {r.get('execution_priority')}")
            print(f"  Primary actions: {r.get('primary_actions', [])}")
        output_file_without_rag = save_action_plan(
            sample, query, [], llm_response_without_rag, results_action_dir
        )
        without_rag_path = (
            output_file_without_rag.parent
            / output_file_without_rag.name.replace("action_plan_", "action_plan_noRAG_")
        )
        output_file_without_rag.rename(without_rag_path)
        if verbose:
            print(f"Saved without-RAG result to {without_rag_path.name}")
    elif verbose:
        print("Failed to get LLM response without RAG")

    if llm_response_with_rag and llm_response_without_rag:
        comparison_file = save_comparison_file(
            sample,
            query,
            rag_results,
            llm_response_with_rag,
            llm_response_without_rag,
            results_action_dir,
            output_file_with_rag,
            without_rag_path,
        )
        if verbose:
            print(f"Saved comparison file to {comparison_file.name}")

    if verbose:
        print("=" * 80)

In [ ]:
# Only run action planning when the model predicts an attack (skip predicted BENIGN).
def _is_predicted_attack(sample: Dict[str, Any]) -> bool:
    pl = str(sample.get("predicted_label") or "").strip().upper()
    return bool(pl) and pl != "BENIGN"


samples_for_actions = [s for s in predictions_data if _is_predicted_attack(s)]
n_skip = len(predictions_data) - len(samples_for_actions)
print(
    f"Action pipeline: {len(samples_for_actions)} sample(s) with predicted attack; "
    f"skipping {n_skip} predicted benign."
)

for sample in samples_for_actions:
    run_agent_pipeline_for_sample(
        sample,
        vector_store,
        attack_actions_data,
        agentic_features_data,
        RESULTS_DIR,
        top_k_retrieve=10,
        verbose=True,
    )

print("Done. Action plans in:", RESULTS_DIR)